In [3]:
from random import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import batman
from transitleastsquares import transitleastsquares
import seaborn as sns
import os
from functools import lru_cache
from itertools import product
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from itertools import groupby
from operator import itemgetter

# ------------------------------------
# 1. Cached light curve loader
# ------------------------------------
@lru_cache(maxsize=10)
def get_lightcurve(tic):
    lc_collection = lk.search_lightcurve(tic, mission="TESS", author="TESS-SPOC", cadence="long").download_all(quality_bitmask="hard")
    if lc_collection is None or len(lc_collection) == 0:
        return None
    
    lc = lc_collection.stitch().remove_nans()

    #plt.plot(lc.time.value, lc.flux.value, 'k.', markersize=1, alpha=0.5)


    return lc.to_pandas().reset_index()

# ------------------------------------
# 2. Flare masking: ±3hr from >5σ events
# ------------------------------------
def mask_flares(df):
    median = df['flux'].median()
    std = df['flux'].std()
    threshold = median + 5 * std
    flare_times = df.loc[df['flux'] > threshold, 'time'].values

    def within_3hr(t):
        return np.any(np.abs(flare_times - t) <= 0.125)

    flare_mask = df['time'].apply(within_3hr)
    return df[~flare_mask].reset_index(drop=True)

# ------------------------------------
# 3. Inject transit only (returns multiplier)
# ------------------------------------
def inject_model(time, period_bin, rp_earth_bin, st_radius, st_mass, st_teff):
    # Constants
    R_earth = 6.371e6        # meters
    R_sun = 6.957e8          # meters
    M_sun = 1.989e30         # kg
    G = 6.67430e-11          # m^3 / kg / s^2

    #np.random.seed(2)
    rp_earth = np.random.uniform(rp_earth_bin[0], rp_earth_bin[1])
    period = np.random.uniform(period_bin[0], period_bin[1])
    print(f"Injected planet radius (Earth Radii): {rp_earth:.2f}, period (days): {period:.2f}")

    # Convert radius: Earth radii to stellar radii
    rp_rstar = (rp_earth * R_earth) / (st_radius * R_sun)

    # Convert mass to kg and period to seconds
    period_sec = period * 86400
    st_mass_kg = st_mass * M_sun
    st_radius_m = st_radius * R_sun

    # Compute semi-major axis in meters
    a_meters = ((G * st_mass_kg * period_sec**2) / (4 * np.pi**2))**(1 / 3)

    # Convert semi-major axis to stellar radii
    a_rstar = a_meters / st_radius_m

    impact_param_upper = .9
    impact_param_lower = 0
    inclination_upper = np.degrees(np.arccos(impact_param_upper * st_radius_m / a_meters))
    inclination_lower = np.degrees(np.arccos(impact_param_lower * st_radius_m / a_meters))
    print(f"Inclination range: {inclination_lower:.2f} to {inclination_upper:.2f} degrees")
    #randomized inclination
    inclination =np.random.uniform(inclination_lower, inclination_upper)
    print(f"rp/rstar^2: {rp_rstar**2:.6f}, a/rstar: {a_rstar:.2f}, period: {period:.2f} days, inclination: {inclination:.2f} degrees")


    #LD LAWS from limb_darkening_laws_investigation.ipynb
    # Quadratic limb darkening coefficients from provided table
    if st_teff < 3500:
        u1, u2 = 0.1598, 0.4534
    elif 3500 <= st_teff < 4000:
        u1, u2 = 0.2366, 0.3750
    elif 4000 <= st_teff < 4500:
        u1, u2 = 0.4540, 0.2136
    elif 4500 <= st_teff < 5000:
        u1, u2 = 0.4672, 0.1826
    elif 5000 <= st_teff < 5500:
        u1, u2 = 0.4068, 0.2063
    elif 5500 <= st_teff < 6000:
        u1, u2 = 0.3632, 0.2209
    elif 6000 <= st_teff < 6500:
        u1, u2 = 0.3273, 0.2323
    elif 6500 <= st_teff < 7000:
        u1, u2 = 0.3101, 0.2284
    elif 7000 <= st_teff:
        u1, u2 = 0.2747, 0.2456

        

    t0 =  np.random.uniform(0, period)
    #t0 = np.random.uniform(time.min() + 1, time.max() - 1)
    print(time.min(), time.max(), t0)


    # Set up transit parameters
    mparams = batman.TransitParams()
    mparams.t0 = t0
    mparams.per = period
    mparams.rp = rp_rstar
    mparams.a = a_rstar
    mparams.inc = inclination
    mparams.ecc = 0
    mparams.w = 90
    mparams.u = [u1, u2]
    mparams.limb_dark = "quadratic"
    #mparams.supersample_factor = 10
    #mparams.exp_time = .016  # 2 min in days
    supersample_factor = 3
    exp_time = .000002

    model = batman.TransitModel(mparams, time)#, supersample_factor=1000, exp_time=.000001)  # 2 min in days
    #print(len(model.light_curve(mparams)))
    #plt.plot(time[:14000], model.light_curve(mparams)[:14000], color='red', alpha=0.5, label='Injected Transit')

    # # Calculate number of points in the first transit (below 1)
    # model_flux = model.light_curve(mparams)
    # in_transit = model_flux < 1
    # # Find the first contiguous block of in-transit points

    # indices = np.where(in_transit)[0]
    # if len(indices) > 0:
    #     # Find contiguous groups
    #     groups = []
    #     for k, g in groupby(enumerate(indices), lambda ix: ix[0] - ix[1]):
    #         group = list(map(itemgetter(1), g))
    #         groups.append(group)
    #     # Take the first group as the first transit
    #     first_transit_indices = groups[0]
    #     n_points_first_transit = len(first_transit_indices)
    # else:
    #     n_points_first_transit = 0

    # print(f"Number of points in first transit (flux < 1): {n_points_first_transit}")

    # # Calculate transit duration in days
    # if n_points_first_transit > 1:
    #     transit_duration = time[first_transit_indices[-1]] - time[first_transit_indices[0]]
    # else:
    #     transit_duration = 0
    # print(f"Transit duration (minutes): {transit_duration*24*60:.2f}")
    # #model = batman.TransitModel(mparams, time, supersample_factor=supersample_factor, exp_time=exp_time)  # 2 min in days

    # #plt.scatter(time[:140], model.light_curve(mparams)[:140], color='blue', alpha=0.5, label=f'Supersampled Injected Transit, ss factor={supersample_factor}, exptime={exp_time}')

    # # rebin time and model to x10 cadence and plot
    # # Calculate original cadence


    # # Rebin the model light curve to simulate lower cadence (e.g., TESS 2-min to 20-min)
    # def rebin(nbin, time, flux):
    #     # Calculate the original time interval between data points
    #     start_interval = time[1] - time[0]
    #     # Set the new bin interval (e.g., 10x the original cadence)
    #     bin_interval = start_interval * nbin
    #     flux_bin = []
    #     time_bin = []
    #     # Initialize tmax to the minimum time value
    #     tmax = np.min(time)
    #     i = 0
    #     # Loop until the end of the time array is reached
    #     while tmax < np.max(time):
    #         # Calculate the center of the current bin
    #         bincenter = np.min(time) + (i + 0.5) * bin_interval
    #         #print(f"Bin center: {bincenter}")
    #         # Find indices of points within half a bin interval of the bin center
    #         index = np.where(np.abs(time - bincenter) < bin_interval / 2.0)
    #         #print(f"Number of points in bin {i}: {len(time[index])}")
    #         # If there are points in this bin, compute the mean flux and time
    #         if len(time[index]):
    #             flux_bin = np.append(flux_bin, np.mean(flux[index]))
    #             time_bin = np.append(time_bin, np.mean(time[index]))
    #             # Update tmax to the maximum time in the current bin
    #             tmax = np.max(time[index])
    #         i = i + 1

    #     return time_bin, flux_bin
    

    # time_rebin, flux_rebin = rebin(10,time,model.light_curve(mparams))
    # print(f"Original cadence: {(time[1]-time[0])*24*60:.2f} min, Rebin cadence: {(time_rebin[1]-time_rebin[0])*24*60:.2f} min")
    # print(f"Original len: {len(time)}, Rebin len: {len(time_rebin)}")


    # # Plot rebinned model
    # plt.plot(time_rebin[:14000], flux_rebin[:14000], color='blue', alpha=0.5, label='Rebinned Transit x10')

    #plt.xlim(1650,1700)
    #plt.xlim(1655,1656)
    #plt.xlim(2445,2460)
    #plot just the first sector
    #plt.legend()
    # plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    # plt.show()
    # plt.plot(model.light_curve(mparams)[:1000], color='red', alpha=0.5, label='Injected Transit')
    # plt.show()
    return model.light_curve(mparams), t0, inclination, rp_earth, period


# 4. Run TLS detection for a single injection
# ------------------------------------
def run_tls(tic, time, flux, true_period):
    # cadence_days = time[1] - time[0]
    # duration_days = 2
    # wl = int(np.round(duration_days / cadence_days))
    # if wl % 2 == 0:
    #     wl += 1  # make it odd

    # print(wl)
    # flux_filter = savgol_filter(flux, wl, 2)
    # flux = flux / flux_filter

    ##LC PLOT
    #plt.plot(time, flux, 'k.', markersize=1, alpha=0.5)
    #plt.title(f"Injected Lightcurve for {tic} flattened")
    #plt.xlim(1650,1700)
    #plt.show()

    if len(time) == 0 or len(flux) == 0:
        return False
    tls = transitleastsquares(time, flux)
    result = tls.power(period_max = 25)

    powers = result.power
    periods = result.periods

    periodogram = pd.DataFrame({
        'period': periods,
        'power': powers
    })

    power_7_periods = periodogram[periodogram['power'] > 7]['period'].values
    print(f"TLS Periods with power > 7: {power_7_periods}")
    power_7_power = periodogram[periodogram['power'] > 7]['power'].values


    print(f"{tic}, FAP: {result.FAP}, SDE: {result.SDE}, True Period: {true_period}")
    # Mark as found if period matches true period or a simple alias (1/2x, 2x, 1/3x, 3x, etc.)
    aliases = [true_period, true_period / 2, true_period * 2, true_period / 3, true_period * 3]
    #found = result.FAP < 1e-4 and any(0.99 * alias < result.period < 1.01 * alias for alias in aliases)
    found = any(0.99 * alias < hp < 1.01 * alias for hp in power_7_periods for alias in aliases)


    alias_found = any(0.99 * alias < result.period < 1.01 * alias for alias in aliases)
    print(f"Detection found: {found}")
    if alias_found == True and found == False:
        print(f"True transit or alias found, but high FAP: {alias_found}")
    #print("TLS Transit duration (hours):", result.duration * 24)


    # #TLS phase fold
    # plt.figure()
    # plt.scatter(result.folded_phase, result.folded_y, color='blue', s=10, alpha=0.5,)
    # plt.plot(result.model_folded_phase, result.model_folded_model, color='red')
    # plt.title(f"TLS Phase Folded Lightcurve for {tic}, TOI {toi}")
    # plt.xlabel('Phase')
    # plt.ylabel('Relative flux');
    # plt.show()

    #Periodogram
    # plt.figure()
    # ax = plt.gca()
    # ax.axvline(result.period, alpha=0.4, lw=3)
    # plt.xlim(np.min(result.periods), np.max(result.periods))
    # for n in range(2, 10):
    #     ax.axvline(n*result.period, alpha=0.4, lw=1, linestyle="dashed")
    #     ax.axvline(result.period / n, alpha=0.4, lw=1, linestyle="dashed")
    # plt.ylabel(r'SDE')
    # plt.xlabel('Period (days)')
    # plt.plot(result.periods, result.power, color='black', lw=0.5)
    # plt.xlim(0, max(result.periods));
    # #plt.xlim(20.8, 21.0)
    # plt.show()


    

    return found, alias_found, result.period, result.SDE, power_7_periods, power_7_power, result.period_uncertainty, result.depth, result.rp_rs, result.FAP, result.snr


# ------------------------------------
# 5. Run detection across grid 
# ------------------------------------
def run_detection_for_tic(tic, periods_bins, radii_bins, output_folder, plots=False, xlim=None):
    
    lc = get_lightcurve(tic)
    if lc is None:
        print(f"❌ No light curve for {tic}")
        return pd.DataFrame()

    base_df = lc[['time', 'flux']].copy()
    

    raw_time = base_df['time'].values
    raw_flux = base_df['flux'].values
    if plots == True:
        plt.plot(raw_time, raw_flux, 'k.', markersize=1, alpha=0.5)
        plt.title(f"Bare Lightcurve for {tic}")
        if xlim is not None:
            plt.xlim(xlim)
        plt.show()
    
   

    # Get stellar parameters
    tic_id = int(tic.split()[1])
    star_params = pd.read_csv("take_two_stellar_params_CTL.csv").query(f"id == {tic_id}")
    if star_params.empty:
        print(f"⚠️ No stellar params for {tic}")
        return pd.DataFrame()

    st_radius = star_params['RAD'].values[0]
    st_mass = star_params['MASS'].values[0] 
    st_teff = star_params['teff'].values[0]
    st_tmag = star_params['tmag'].values[0]

    results = pd.DataFrame()
     # Loop over period-radius grid
    for period_bin, rp_earth_bin in product(periods_bins, radii_bins):
        detections = 0
        for trial in range(trials):
            print(f"Running trial {trial + 1}/{trials} for {tic}, Period: {period_bin}, Rp (Earth Radii): {rp_earth_bin}")
            # plt.plot(raw_time, raw_flux, 'k.', markersize=1, alpha=0.5)
            # plt.title(f"Bare Lightcurve for {tic}")
            # #plt.xlim(1650,1700)
            # plt.show()
            print(tic)
            transit_signal, true_t0, true_inc, true_rp, true_period = inject_model(raw_time, period_bin, rp_earth_bin, st_radius, st_mass, st_teff)
            injected_flux = raw_flux * transit_signal
            injected_flux_mean_normalized = injected_flux #/ np.mean(injected_flux)
            injected_df_mean_normalized = pd.DataFrame({'time': raw_time, 'flux': injected_flux_mean_normalized})
            flux_normalized_flares_removed = mask_flares(injected_df_mean_normalized)

            if plots == True:
                plt.plot(flux_normalized_flares_removed['time'], flux_normalized_flares_removed['flux'], 'k.', markersize=1, alpha=0.5)
                plt.title(f"Injected Lightcurve for tic {tic} unflattened")
                if xlim is not None:
                    plt.xlim(xlim)
                plt.show()

            lc = lk.LightCurve(time=flux_normalized_flares_removed['time'], flux=flux_normalized_flares_removed['flux'])
            time = lc.time.value  # in days
            if len(time) < 2:
                return None

            cadence_days = time[1] - time[0]
            duration_days = 2
            wl = int(np.round(duration_days / cadence_days))
            if wl % 2 == 0:
                wl += 1  # make it odd, as required

            #print(f"Cadence: {cadence_days:.5f} days, window_length: {wl}")

            lc_flat = lc.flatten(window_length=wl, polyorder=2)
            flat_df = lc_flat.to_pandas().reset_index()
            time = flat_df['time'].values
            flux = flat_df['flux'].values
            #flux = flux / np.median(flux)
            if plots == True:
                plt.plot(time, flux, 'k.', markersize=1, alpha=0.5)
                plt.title(f"Injected Lightcurve for {tic} flattened")
                if xlim is not None:
                    plt.xlim(xlim)
                plt.show()

            lightcurve_data = pd.DataFrame({'time': time, 'flux': flux, 'true_t0': true_t0, 'true_inc': true_inc, 'true_rp_earth': true_rp, 'true_period': true_period})
            file_name = f'{output_folder}/P{true_period:.3f}_Rp{true_rp:.3f}_trial{trial}.csv'
            os.makedirs(output_folder, exist_ok=True)
            lightcurve_data.to_csv(file_name, index=False)

            
        
            




# ------------------------------------
# 6. Save heatmap and results
# ------------------------------------
# def save_heatmap(data, tic):
#     data = data[['True Radius (Earth Radii)', 'True Period (Days)', 'Detection %']]
#     pivot = data.pivot(index="True Radius (Earth Radii)", columns="True Period (Days)", values="Detection %")
#     plt.figure(figsize=(8, 6))
#     sns.heatmap(pivot, cmap="viridis", vmin=0, vmax=1, cbar_kws={'label': 'Percent found'}).invert_yaxis()
#     plt.title(f"{tic} noise transit detection rate")
#     plt.xlabel("Period (Days)")
#     plt.ylabel("Radius (Earth Radii)")
#     plt.tight_layout()
#     os.makedirs("heatmap_data", exist_ok=True)
#     plt.savefig(f'heatmap_data/{tic.replace(" ", "_")}_cel_heatmap.png', dpi=300)
    

#     path = f'heatmap_data/{tic.replace(" ", "_")}_cel_heatmap.csv'
#     data.to_csv(path, index=False)
#     print(f"✅ Saved: {path}")
#     plt.show()
    


# ------------------------------------
# 7. Main driver
# ------------------------------------
if __name__ == "__main__":

    #save_file = "./inj_results/all_results_to_25P_multi_savgolay_2dayfilter_overSDE7.csv"
    #os.makedirs("inj_rec_results", exist_ok=True)

    #save_file = './inj_rec_results/test_injrec_CTL_completeness_inquiry.csv'

    # Load TIC IDs from the CSV file
    star_params = pd.read_csv("take_two_stellar_params_CTL.csv")
    # star_params = star_params[star_params['id'].isin([
    #     92005981,
    #     337025463,
    #     457925111,
    #     375061802,
    #     74535131,
    #     173535874,
    #     51988363,
    #     172076858,
    #     140067408,
    #     284566637,
    #     11148069
    # ])]
    #print(star_params.head(20))
    #star_params = star_params[5:]
    #star_params = star_params.iloc[:1]  # Limit to first 10 for testing, remove this line for full run
    #star_params = star_params.iloc[11:12] 
    #star_params = star_params[star_params['id'] == 241204178]
    #for each pair of toi and tic, there's a true period and rp_earth to run

    df_save = pd.DataFrame()

    
    for _, row in star_params.iterrows():
        tic = f"TIC {int(row['id'])}"
        #print(tic)
        output_folder = f'./take_two_injected_flattened_lcs_t0_fixed/{tic.replace(" ", "_")}'
        if os.path.exists(output_folder):
            print(f"Skipping {tic}, already processed.")
            continue

    #periods_bins = np.exp(np.linspace(-.52, 5.77, num=26))[1:13][::2]
    #radii_bins = np.exp(np.linspace(1.63875, 0.36125, num=14))

    #radii = radii[:1] 


    # Log-spaced list between 1.4 and 5 (inclusive), in log10 space
    radii_centers = np.logspace(np.log10(1.4), np.log10(5), num=6)
    # Log-spaced list between 0.7 and 25 (inclusive), in log10 space
    periods_centers = np.logspace(np.log10(0.7), np.log10(25), num=7)

    # Convert radii_centers and periods_centers to floats with up to 3 decimals
    radii_centers = [float(f"{v:.3f}") for v in radii_centers]
    periods_centers = [float(f"{v:.3f}") for v in periods_centers]
    

    # Generate bin edges from centers
    periods_log = np.log10(periods_centers)
    log_step = periods_log[1] - periods_log[0]
    period_bin_edges = [10**(periods_log[0] - log_step/2)]
    period_bin_edges.extend([10**((periods_log[i] + periods_log[i+1])/2) for i in range(len(periods_log)-1)])
    period_bin_edges.append(10**(periods_log[-1] + log_step/2))
    periods_bins = [(period_bin_edges[i], period_bin_edges[i+1]) for i in range(len(period_bin_edges)-1)]

    radii_log = np.log10(radii_centers)
    log_step = radii_log[1] - radii_log[0]
    radius_bin_edges = [10**(radii_log[0] - log_step/2)]
    radius_bin_edges.extend([10**((radii_log[i] + radii_log[i+1])/2) for i in range(len(radii_log)-1)])
    radius_bin_edges.append(10**(radii_log[-1] + log_step/2))
    radii_bins = [(radius_bin_edges[i], radius_bin_edges[i+1]) for i in range(len(radius_bin_edges)-1)]

    print("Period bins:", periods_bins)
    print("Radius bins:", radii_bins)

    trials = 1


    try:
        df_results = run_detection_for_tic(tic, periods_bins, radii_bins, output_folder, plots=False, xlim=None)
        #save_heatmap(df_results, tic)

    except Exception as e:
        print(f"❌ Error occurred for {tic}: {e}")
        # df_results = pd.DataFrame([{
        #     'TIC': tic,
        #     'Stellar Radius': None,
        #     'Stellar Temperature': None,
        #     'Stellar Magnitude': None,
        #     'True Radius (Earth Radii)': None,
        #     'True Period (Days)': None,
        #     'Detection': e,
        #     'TLS Period': None,
        #     'TLS SDE strongest': None,
        #     'TLS SDE > 7 array': None,
        #     'TLS Periods array': None,
        #     'TLS Period Uncertainty': None,
        #     'TLS Depth': None,
        #     'TLS Rp/Rs': None,
        #     'TLS FAP': None,
        #     'TLS SNR': None,
        #     #'Detection %': None,
        #     '# trials': None
        # }])


        # updated = df_results
        # updated.to_csv(save_file, mode='a', index=False, header=False)

        #save_heatmap(df_results, tic)


Skipping TIC 51988363, already processed.
Skipping TIC 172076858, already processed.
Skipping TIC 140067408, already processed.
Skipping TIC 284566637, already processed.
Skipping TIC 11148069, already processed.
Skipping TIC 457925111, already processed.
Skipping TIC 173535874, already processed.
Skipping TIC 92005981, already processed.
Skipping TIC 186968731, already processed.
Skipping TIC 146579468, already processed.
Skipping TIC 335918986, already processed.
Skipping TIC 317880236, already processed.
Skipping TIC 3358442, already processed.
Skipping TIC 36648057, already processed.
Skipping TIC 52093014, already processed.
Skipping TIC 50490788, already processed.
Skipping TIC 123405412, already processed.
Skipping TIC 189400940, already processed.
Skipping TIC 123968788, already processed.
Skipping TIC 335622798, already processed.
Skipping TIC 318190759, already processed.
Skipping TIC 114182302, already processed.
Skipping TIC 137057529, already processed.
Skipping TIC 293406

In [5]:
from collections import Counter
import os

folder_counts = []
base_dir = 'take_two_injected_flattened_lcs_t0_fixed'

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        num_files = len(os.listdir(folder_path))
        folder_counts.append(num_files)

count_summary = Counter(folder_counts)
for num_files, num_folders in sorted(count_summary.items()):
    print(f"{num_folders} folders have {num_files} files")

# Print folders that don't have 42 files
for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        num_files = len(os.listdir(folder_path))
        if num_files != 42:
            print(f"{folder}: {num_files} files")
    
# # Delete folders that don't have 42 files
# for folder in os.listdir(base_dir):
#     folder_path = os.path.join(base_dir, folder)
#     if os.path.isdir(folder_path):
#         num_files = len(os.listdir(folder_path))
#         if num_files != 42:
#             print(f"Deleting {folder} with {num_files} files")
#             for file in os.listdir(folder_path):
#                 os.remove(os.path.join(folder_path, file))
#             os.rmdir(folder_path)

859 folders have 42 files


In [51]:
from collections import Counter
import os

folder_counts = []
base_dir = 'take_two_injected_flattened_lcs_with_extra'

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        num_files = len(os.listdir(folder_path))
        folder_counts.append(num_files)

count_summary = Counter(folder_counts)
for num_files, num_folders in sorted(count_summary.items()):
    print(f"{num_folders} folders have {num_files} files")
    

1 folders have 1 files
575 folders have 42 files
169 folders have 84 files
1 folders have 99 files
1 folders have 125 files
98 folders have 126 files
7 folders have 168 files
1 folders have 170 files
1 folders have 231 files
1 folders have 242 files


In [ ]:
# # Find folders (TICs) with a number of files not equal to 42
# non_42_tics = []
# for folder in os.listdir(base_dir):
#     folder_path = os.path.join(base_dir, folder)
#     if os.path.isdir(folder_path):
#         num_files = len(os.listdir(folder_path))
#         if num_files != 42:
#             non_42_tics.append((folder, num_files))

# for tic, nfiles in non_42_tics:
#     print(f"{tic}: {nfiles} files")
    
# for tic, _ in non_42_tics:
#     folder_path = os.path.join(base_dir, tic)
#     if os.path.isdir(folder_path):
#         print(f"Deleting folder: {folder_path}")
#         for file in os.listdir(folder_path):
#             os.remove(os.path.join(folder_path, file))
#         os.rmdir(folder_path)

TIC_92005981: 170 files
TIC_337025463: 1 files
TIC_457925111: 231 files
TIC_375061802: 99 files
TIC_74535131: 125 files
TIC_173535874: 242 files
Deleting folder: take_two_injected_flattened_lcs/TIC_92005981
Deleting folder: take_two_injected_flattened_lcs/TIC_337025463
Deleting folder: take_two_injected_flattened_lcs/TIC_457925111
Deleting folder: take_two_injected_flattened_lcs/TIC_375061802
Deleting folder: take_two_injected_flattened_lcs/TIC_74535131
Deleting folder: take_two_injected_flattened_lcs/TIC_173535874


In [ ]:
92005981
337025463
457925111
375061802
74535131
173535874
51988363
172076858
140067408
284566637
11148069

Folder for TIC 51988363 exists: False


In [ ]:
# import random

# folders_with_84_files = []
# for folder in os.listdir(base_dir):
#     folder_path = os.path.join(base_dir, folder)
#     if os.path.isdir(folder_path):
#         files = sorted(os.listdir(folder_path))
#         if len(files) == 84:
#             folders_with_84_files.append(folder)
#             # Group every two files
#             grouped = [files[i:i+2] for i in range(0, len(files), 2)]
#             print(f"Folder: {folder}")
#             for pair in grouped:
#                 print(pair)
#                 file_to_delete = random.choice(pair)
#                 os.remove(os.path.join(folder_path, file_to_delete))
#                 print(f"Deleted: {file_to_delete}")



Folder: TIC_150098542
['P0.565_Rp2.523_trial0.csv', 'P0.629_Rp4.818_trial0.csv']
Deleted: P0.629_Rp4.818_trial0.csv
['P0.715_Rp4.207_trial0.csv', 'P0.741_Rp1.896_trial0.csv']
Deleted: P0.741_Rp1.896_trial0.csv
['P0.754_Rp3.403_trial0.csv', 'P0.759_Rp2.768_trial0.csv']
Deleted: P0.754_Rp3.403_trial0.csv
['P0.803_Rp3.976_trial0.csv', 'P0.816_Rp1.927_trial0.csv']
Deleted: P0.816_Rp1.927_trial0.csv
['P0.839_Rp2.195_trial0.csv', 'P0.843_Rp1.278_trial0.csv']
Deleted: P0.839_Rp2.195_trial0.csv
['P0.888_Rp1.398_trial0.csv', 'P0.938_Rp4.495_trial0.csv']
Deleted: P0.888_Rp1.398_trial0.csv
['P1.008_Rp4.734_trial0.csv', 'P1.087_Rp4.961_trial0.csv']
Deleted: P1.008_Rp4.734_trial0.csv
['P1.147_Rp1.845_trial0.csv', 'P1.166_Rp1.238_trial0.csv']
Deleted: P1.147_Rp1.845_trial0.csv
['P1.289_Rp2.544_trial0.csv', 'P1.324_Rp4.048_trial0.csv']
Deleted: P1.324_Rp4.048_trial0.csv
['P1.326_Rp1.627_trial0.csv', 'P1.397_Rp1.548_trial0.csv']
Deleted: P1.326_Rp1.627_trial0.csv
['P1.411_Rp2.457_trial0.csv', 'P1.470_

In [ ]:
# folders_with_126_files = []
# for folder in os.listdir(base_dir):
#     folder_path = os.path.join(base_dir, folder)
#     if os.path.isdir(folder_path):
#         files = sorted(os.listdir(folder_path))
#         if len(files) == 126:
#             folders_with_126_files.append(folder)
#             # Group every three files
#             grouped = [files[i:i+3] for i in range(0, len(files), 3)]
#             print(f"Folder: {folder}")
#             for triplet in grouped:
#                 print(triplet)
#                 files_to_delete = random.sample(triplet, 2)
#                 for file_to_delete in files_to_delete:
#                     os.remove(os.path.join(folder_path, file_to_delete))
#                     print(f"Deleted: {file_to_delete}")

Folder: TIC_306398611
['P0.580_Rp5.067_trial0.csv', 'P0.590_Rp4.939_trial0.csv', 'P0.643_Rp2.218_trial0.csv']
Deleted: P0.590_Rp4.939_trial0.csv
Deleted: P0.580_Rp5.067_trial0.csv
['P0.653_Rp4.294_trial0.csv', 'P0.662_Rp3.215_trial0.csv', 'P0.665_Rp1.333_trial0.csv']
Deleted: P0.662_Rp3.215_trial0.csv
Deleted: P0.653_Rp4.294_trial0.csv
['P0.669_Rp3.297_trial0.csv', 'P0.691_Rp2.430_trial0.csv', 'P0.706_Rp1.579_trial0.csv']
Deleted: P0.691_Rp2.430_trial0.csv
Deleted: P0.706_Rp1.579_trial0.csv
['P0.724_Rp2.624_trial0.csv', 'P0.768_Rp3.339_trial0.csv', 'P0.775_Rp1.828_trial0.csv']
Deleted: P0.768_Rp3.339_trial0.csv
Deleted: P0.724_Rp2.624_trial0.csv
['P0.838_Rp1.681_trial0.csv', 'P0.841_Rp3.453_trial0.csv', 'P0.873_Rp4.757_trial0.csv']
Deleted: P0.841_Rp3.453_trial0.csv
Deleted: P0.838_Rp1.681_trial0.csv
['P0.893_Rp1.260_trial0.csv', 'P0.895_Rp3.815_trial0.csv', 'P0.929_Rp1.759_trial0.csv']
Deleted: P0.893_Rp1.260_trial0.csv
Deleted: P0.929_Rp1.759_trial0.csv
['P0.962_Rp3.173_trial0.csv', 

In [ ]:
# folders_with_168_files = []
# for folder in os.listdir(base_dir):
#     folder_path = os.path.join(base_dir, folder)
#     if os.path.isdir(folder_path):
#         files = sorted(os.listdir(folder_path))
#         if len(files) == 168:
#             folders_with_168_files.append(folder)
#             # Group every four files alphabetically
#             grouped = [files[i:i+4] for i in range(0, len(files), 4)]
#             print(f"Folder: {folder}")
#             for group in grouped:
#                 print(group)
#                 files_to_delete = random.sample(group, 3)
#                 for file_to_delete in files_to_delete:
#                     os.remove(os.path.join(folder_path, file_to_delete))
#                     print(f"Deleted: {file_to_delete}")

Folder: TIC_146579468
['P0.536_Rp5.424_trial0.csv', 'P0.550_Rp2.847_trial0.csv', 'P0.561_Rp1.797_trial0.csv', 'P0.571_Rp1.321_trial0.csv']
Deleted: P0.561_Rp1.797_trial0.csv
Deleted: P0.536_Rp5.424_trial0.csv
Deleted: P0.571_Rp1.321_trial0.csv
['P0.581_Rp4.825_trial0.csv', 'P0.590_Rp4.018_trial0.csv', 'P0.632_Rp4.187_trial0.csv', 'P0.635_Rp2.750_trial0.csv']
Deleted: P0.632_Rp4.187_trial0.csv
Deleted: P0.635_Rp2.750_trial0.csv
Deleted: P0.590_Rp4.018_trial0.csv
['P0.651_Rp1.619_trial0.csv', 'P0.681_Rp5.569_trial0.csv', 'P0.690_Rp2.708_trial0.csv', 'P0.716_Rp2.303_trial0.csv']
Deleted: P0.681_Rp5.569_trial0.csv
Deleted: P0.690_Rp2.708_trial0.csv
Deleted: P0.716_Rp2.303_trial0.csv
['P0.727_Rp1.422_trial0.csv', 'P0.750_Rp2.167_trial0.csv', 'P0.759_Rp1.955_trial0.csv', 'P0.761_Rp2.509_trial0.csv']
Deleted: P0.727_Rp1.422_trial0.csv
Deleted: P0.750_Rp2.167_trial0.csv
Deleted: P0.759_Rp1.955_trial0.csv
['P0.764_Rp3.483_trial0.csv', 'P0.785_Rp1.243_trial0.csv', 'P0.825_Rp3.057_trial0.csv', 'P